# ANACITY Facility Usage Prediction System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaiswalwrites/community-facility-usage-prediction-system/blob/main/Facility_Usage_Prediction_System.ipynb)

An end-to-end Machine Learning pipeline predicting resident facility usage, optimal day & hour, and push notification nudge timing without temporal data leakage.

---

## 1. Install & Import Dependencies

In [ ]:
!pip install -q pandas scikit-learn numpy

import random
import json
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

print("✅ Dependencies successfully loaded!")

## 2. Synthetic Data Synthesis Engine

In [ ]:
def generate_synthetic_bookings(num_residents=120, num_days=180, random_seed=42):
    random.seed(random_seed)
    np.random.seed(random_seed)
    facilities = ["Gym", "Swimming Pool", "Badminton Court", "Tennis Court", "Clubhouse"]
    
    archetypes = [
        {"name": "Morning Gym", "weights": [0.70, 0.10, 0.10, 0.05, 0.05], "preferred_hours": [6, 7, 8], "preferred_days": [0, 1, 2, 3, 4], "avg_lead_hours": 14.0, "frequency": 0.65},
        {"name": "Evening Badminton", "weights": [0.10, 0.05, 0.70, 0.10, 0.05], "preferred_hours": [18, 19, 20], "preferred_days": [0, 1, 2, 3, 4], "avg_lead_hours": 28.0, "frequency": 0.50},
        {"name": "Weekend Pool", "weights": [0.10, 0.55, 0.05, 0.10, 0.20], "preferred_hours": [10, 11, 15, 16, 17], "preferred_days": [5, 6], "avg_lead_hours": 6.0, "frequency": 0.40},
        {"name": "Tennis Specialist", "weights": [0.10, 0.05, 0.10, 0.70, 0.05], "preferred_hours": [7, 8, 17, 18], "preferred_days": [1, 3, 5, 6], "avg_lead_hours": 42.0, "frequency": 0.35},
        {"name": "Clubhouse Social", "weights": [0.05, 0.15, 0.10, 0.10, 0.60], "preferred_hours": [14, 15, 18, 19, 20], "preferred_days": [4, 5, 6], "avg_lead_hours": 20.0, "frequency": 0.30}
    ]

    residents = [{"id": f"R-{100 + i}", "archetype": random.choice(archetypes)} for i in range(num_residents)]
    start_date = datetime.strptime("2026-01-01", "%Y-%m-%d")
    bookings = []

    for day_offset in range(num_days):
        current_day = start_date + timedelta(days=day_offset)
        day_of_week = current_day.weekday()
        for res in residents:
            arch = res["archetype"]
            day_mult = 1.6 if day_of_week in arch["preferred_days"] else 0.4
            prob = arch["frequency"] * day_mult * random.uniform(0.7, 1.3)
            if random.random() < prob:
                facility = random.choice(facilities) if random.random() < 0.10 else np.random.choice(facilities, p=arch["weights"])
                usage_hour = random.randint(6, 21) if random.random() < 0.15 else random.choice(arch["preferred_hours"])
                usage_dt = current_day.replace(hour=usage_hour, minute=0, second=0)
                lead_hours = max(0.5, np.random.normal(arch["avg_lead_hours"], arch["avg_lead_hours"] * 0.25))
                booking_dt = usage_dt - timedelta(hours=lead_hours)
                bookings.append({
                    "resident_id": res["id"],
                    "facility_name": facility,
                    "booking_timestamp": booking_dt.strftime("%Y-%m-%d %H:%M:%S"),
                    "usage_timestamp": usage_dt.strftime("%Y-%m-%d %H:%M:%S")
                })

    df = pd.DataFrame(bookings)
    df["booking_dt"] = pd.to_datetime(df["booking_timestamp"])
    df = df.sort_values("booking_dt").drop(columns=["booking_dt"]).reset_index(drop=True)
    return df

raw_df = generate_synthetic_bookings()
print(f"✅ Generated {len(raw_df)} realistic booking records.")
raw_df.head()

## 3. Leakage-Safe Feature Engineering

In [ ]:
FACILITIES = ["Gym", "Swimming Pool", "Badminton Court", "Tennis Court", "Clubhouse"]

def extract_features(df, min_history=2):
    df = df.copy()
    df["booking_dt"] = pd.to_datetime(df["booking_timestamp"])
    df["usage_dt"] = pd.to_datetime(df["usage_timestamp"])
    df["target_facility"] = df["facility_name"]
    df["target_usage_day"] = df["usage_dt"].dt.dayofweek
    df["target_usage_hour"] = df["usage_dt"].dt.hour
    df["target_lead_hours"] = (df["usage_dt"] - df["booking_dt"]).dt.total_seconds() / 3600.0
    df = df.sort_values("booking_dt").reset_index(drop=True)
    
    features_list = []
    for resident_id, group in df.groupby("resident_id"):
        history = []
        for record in group.to_dict("records"):
            cur_dt = record["booking_dt"]
            past = [h for h in history if h["booking_dt"] < cur_dt]
            if len(past) >= min_history:
                hist_len = len(past)
                fac_counts = {f: 0 for f in FACILITIES}
                for h in past: fac_counts[h["facility_name"]] += 1
                fac_ratios = {f"ratio_{f}": fac_counts[f] / hist_len for f in FACILITIES}
                top_fac = max(fac_counts, key=fac_counts.get)
                
                day_counts = [0] * 7
                for h in past: day_counts[h["target_usage_day"]] += 1
                day_ratios = {f"ratio_day_{d}": day_counts[d] / hist_len for d in range(7)}
                
                hours = [h["target_usage_hour"] for h in past]
                leads = [h["target_lead_hours"] for h in past]
                last_1 = past[-1]
                last_2 = past[-2] if hist_len >= 2 else last_1
                
                feat = {
                    "resident_id": resident_id,
                    "booking_timestamp": record["booking_timestamp"],
                    "usage_timestamp": record["usage_timestamp"],
                    "target_facility": record["target_facility"],
                    "target_usage_day": record["target_usage_day"],
                    "target_usage_hour": record["target_usage_hour"],
                    "target_lead_hours": record["target_lead_hours"],
                    "hist_count": hist_len,
                    "top_facility": top_fac,
                    "top_day": int(np.argmax(day_counts)),
                    "avg_usage_hour": np.mean(hours),
                    "std_usage_hour": np.std(hours) if hist_len > 1 else 0.0,
                    "avg_lead_hours": np.mean(leads),
                    "days_since_last_booking": (cur_dt - last_1["booking_dt"]).total_seconds() / 86400.0,
                    "lag1_facility": last_1["facility_name"],
                    "lag1_usage_day": last_1["target_usage_day"],
                    "lag1_usage_hour": last_1["target_usage_hour"],
                    "lag2_facility": last_2["facility_name"]
                }
                feat.update(fac_ratios)
                feat.update(day_ratios)
                features_list.append(feat)
            history.append(record)
            
    return pd.DataFrame(features_list)

feat_df = extract_features(raw_df)
print(f"✅ Extracted leakage-safe feature matrix: {feat_df.shape}")

## 4. Train/Test Split & Multi-Output Model Pipeline

In [ ]:
# Chronological Holdout Split (80% Train, 20% Unseen Test)
split_idx = int(len(feat_df) * 0.80)
train_df = feat_df.iloc[:split_idx].copy()
test_df = feat_df.iloc[split_idx:].copy()

feature_cols = [
    "hist_count", "avg_usage_hour", "std_usage_hour", "avg_lead_hours",
    "days_since_last_booking", "top_day", "lag1_usage_day", "lag1_usage_hour",
    "top_facility_enc", "lag1_facility_enc", "lag2_facility_enc",
    "ratio_Gym", "ratio_Swimming Pool", "ratio_Badminton Court", "ratio_Tennis Court", "ratio_Clubhouse",
    "ratio_day_0", "ratio_day_1", "ratio_day_2", "ratio_day_3", "ratio_day_4", "ratio_day_5", "ratio_day_6"
]

enc = LabelEncoder()
enc.fit(FACILITIES)

def prep_matrix(d):
    d = d.copy()
    d["top_facility_enc"] = enc.transform(d["top_facility"])
    d["lag1_facility_enc"] = enc.transform(d["lag1_facility"])
    d["lag2_facility_enc"] = enc.transform(d["lag2_facility"])
    return d[feature_cols]

X_train = prep_matrix(train_df)
X_test = prep_matrix(test_df)

fac_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_train, enc.transform(train_df["target_facility"]))
day_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_train, train_df["target_usage_day"])
hour_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_train, train_df["target_usage_hour"])
lead_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42).fit(X_train, train_df["target_lead_hours"])

print("✅ Pipeline fitted on 80% train holdout dataset!")

## 5. Evaluation & Prediction Review Output Table

In [ ]:
pred_fac_enc = fac_model.predict(X_test)
pred_facs = enc.inverse_transform(pred_fac_enc)
pred_days = day_model.predict(X_test)
pred_hours = hour_model.predict(X_test)
pred_leads = lead_model.predict(X_test)

results = test_df.copy()
results["pred_facility"] = pred_facs
results["pred_usage_day"] = pred_days
results["pred_usage_hour"] = pred_hours
results["pred_lead_hours"] = pred_leads

results["fac_match"] = results["pred_facility"] == results["target_facility"]
results["day_match"] = results["pred_usage_day"] == results["target_usage_day"]
results["hour_match"] = (results["pred_usage_hour"] - results["target_usage_hour"]).abs() <= 1

results["actual_booked_dt"] = pd.to_datetime(results["booking_timestamp"])
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

pred_nudge_times = []
for idx, row in results.iterrows():
    booking_dt = pd.to_datetime(row["booking_timestamp"])
    days_ahead = (int(row["pred_usage_day"]) - booking_dt.weekday()) % 7
    if days_ahead == 0 and row["pred_usage_hour"] <= booking_dt.hour:
        days_ahead = 7
    pred_use_dt = (booking_dt + timedelta(days=days_ahead)).replace(hour=int(row["pred_usage_hour"]), minute=0)
    pred_nudge_dt = pred_use_dt - timedelta(hours=float(row["pred_lead_hours"]))
    pred_nudge_times.append(pred_nudge_dt.strftime("%Y-%m-%d %H:%M"))

results["pred_nudge_timestamp"] = pred_nudge_times
results["pred_nudge_dt"] = pd.to_datetime(pred_nudge_times)
nudge_diffs = (results["pred_nudge_dt"] - results["actual_booked_dt"]).abs().dt.total_seconds() / 3600.0
results["nudge_match"] = nudge_diffs <= 2.5

results["match_score"] = results["fac_match"].astype(int) + results["day_match"].astype(int) + results["hour_match"].astype(int) + results["nudge_match"].astype(int)
results["exact_match"] = results["match_score"] == 4

print("=========================================================")
print("               EVALUATION METRICS SUMMARY               ")
print("=========================================================")
print(f"Facility Accuracy           : {results['fac_match'].mean()*100:.2f}%")
print(f"Usage Day Accuracy          : {results['day_match'].mean()*100:.2f}%")
print(f"Usage Hour Accuracy (±1h)   : {results['hour_match'].mean()*100:.2f}%")
print(f"Nudge Time Accuracy (±2.5h) : {results['nudge_match'].mean()*100:.2f}%")
print(f"Nudge Time MAE              : {nudge_diffs.mean():.2f} hours")
print(f"---------------------------------------------------------")
print(f"OVERALL EXACT 4/4 MATCH RATE: {results['exact_match'].mean()*100:.2f}%")
print("=========================================================")

## 6. Formatted Section 3.5 Prediction Review Table Output

In [ ]:
review_rows = []
for idx, row in results.reset_index().iterrows():
    res_id = row["resident_id"]
    past_summary = f"{row['lag1_facility']} / {DAY_NAMES[int(row['lag1_usage_day'])]} / {int(row['lag1_usage_hour']):02d}:00 | {row['top_facility']} (Top)"
    pred_str = f"{row['pred_facility']} / {DAY_NAMES[int(row['pred_usage_day'])]} / {int(row['pred_usage_hour']):02d}:00 | Nudge {DAY_NAMES[row['pred_nudge_dt'].weekday()]} {row['pred_nudge_dt'].strftime('%H:%M')}"
    actual_str = f"{row['target_facility']} / {DAY_NAMES[int(row['target_usage_day'])]} / {int(row['target_usage_hour']):02d}:00 | Booked {DAY_NAMES[row['actual_booked_dt'].weekday()]} {row['actual_booked_dt'].strftime('%H:%M')}"
    
    match_indicator = "YES" if row["exact_match"] else "NO"
    review_rows.append({
        "REF / RESIDENT": f"{res_id}-#{idx+1}",
        "PAST BOOKINGS": past_summary,
        "PREDICTION": pred_str,
        "ACTUAL": actual_str,
        "MATCH": f"{match_indicator} ({row['match_score']}/4)"
    })

review_df = pd.DataFrame(review_rows)
display(review_df.head(15))